# Configuration

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests", "psycopg2-binary", "kafka-python-ng"])
print("requests + psycopg2 + kafka-python-ng ready")

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
import os, time, json, requests
import psycopg2
from kafka.admin import KafkaAdminClient

S3_ENDPOINT = "http://minio:9000"
S3_BUCKET   = "s3a://warehouse"
BOOTSTRAP   = "kafka:9092"

# Credentials from .env file
MINIO_ROOT_USER = os.environ.get("MINIO_ROOT_USER", "admin")
MINIO_ROOT_PASSWORD = os.environ.get("MINIO_ROOT_PASSWORD", "admin123")

# PostgreSQL credentials from .env file
PG_HOST = os.environ.get("PG_HOST", "postgres")
PG_PORT = int(os.environ.get("PG_PORT", 5432))
PG_DB = os.environ.get("PG_DB", "sourcedb")
PG_USER = os.environ.get("PG_USER", "cdc_user")
PG_PASSWORD = os.environ.get("PG_PASSWORD", "cdc_pass")

# Spark Session Configuration
spark = (
    SparkSession.builder
    .appName("bdm_project_3")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    
    # MinIO / S3 configuration
    .config("spark.hadoop.fs.s3a.endpoint", S3_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ROOT_USER)      
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_ROOT_PASSWORD)   
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    
    # Iceberg configuration
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "hadoop")
    .config("spark.sql.catalog.lakehouse.warehouse", S3_BUCKET)
    .config("spark.sql.defaultCatalog", "lakehouse")
    
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark {spark.version}")
print(f"Kafka: {BOOTSTRAP}")
print(f"S3:    {S3_ENDPOINT}")
print(f"MinIO User: {MINIO_ROOT_USER}")
print("Ready!")

PG_CONN = dict(host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PASSWORD)

def pg_execute(sql, fetch=False):
    conn = psycopg2.connect(**PG_CONN)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(sql)
    result = cur.fetchall() if fetch else None
    cur.close()
    conn.close()
    return result

ver = pg_execute("SELECT version();", fetch=True)
print(f"PostgreSQL: {ver[0][0][:60]}...")

wal = pg_execute("SHOW wal_level;", fetch=True)
print(f"wal_level = {wal[0][0]}")
assert wal[0][0] == "logical", "wal_level must be 'logical' for CDC!"

requests + psycopg2 + kafka-python-ng ready
Spark 4.1.0
Kafka: kafka:9092
S3:    http://minio:9000
MinIO User: admin
Ready!
PostgreSQL: PostgreSQL 16.13 on x86_64-pc-linux-musl, compiled by gcc (A...
wal_level = logical


# 1. Debezium CDC connector

Snapshot Mode Configuration:

Setting: `snapshot.mode: initial`

Reason: This ensures that downstream layers are initialized with a complete and consistent baseline of the source data, rather than only capturing changes that occur after the pipeline starts. Without this, existing records would be missing, leading to incomplete datasets.

In [2]:
CONNECT_URL = "http://connect:8083"

for i in range(30):
    try:
        r = requests.get(f"{CONNECT_URL}/")
        if r.status_code == 200:
            print(f"Kafka Connect is ready: {r.json()}")
            break
    except:
        pass
    print(f"Waiting for Kafka Connect... ({i+1})")
    time.sleep(3)
else:
    raise RuntimeError("Kafka Connect did not start in time!")

Kafka Connect is ready: {'version': '3.9.0', 'commit': 'a60e31147e6b01ee', 'kafka_cluster_id': 'MkU3OEVBNTcwNTJENDM2Qg'}


In [3]:
requests.delete(f"{CONNECT_URL}/connectors/pg-cdc-connector")
time.sleep(2)

connector_config = {
    "name": "pg-cdc-connector",
    "config": {
        "connector.class": "io.debezium.connector.postgresql.PostgresConnector",
        "database.hostname": PG_HOST,
        "database.port": PG_PORT,
        "database.user": PG_USER,
        "database.password": PG_PASSWORD,
        "database.dbname": PG_DB,
        "topic.prefix": "dbserver1",
        "table.include.list": "public.customers, public.drivers",
        "plugin.name": "pgoutput",
        "snapshot.mode": "initial",
        "key.converter.schemas.enable": "false",
        "value.converter.schemas.enable": "false",
    }
}

r = requests.post(
    f"{CONNECT_URL}/connectors",
    headers={"Content-Type": "application/json"},
    data=json.dumps(connector_config),
)
print(f"Status: {r.status_code}")
print(json.dumps(r.json(), indent=2))

Status: 201
{
  "name": "pg-cdc-connector",
  "config": {
    "connector.class": "io.debezium.connector.postgresql.PostgresConnector",
    "database.hostname": "postgres",
    "database.port": "5432",
    "database.user": "cdc_user",
    "database.password": "cdc_pass",
    "database.dbname": "sourcedb",
    "topic.prefix": "dbserver1",
    "table.include.list": "public.customers, public.drivers",
    "plugin.name": "pgoutput",
    "snapshot.mode": "initial",
    "key.converter.schemas.enable": "false",
    "value.converter.schemas.enable": "false",
    "name": "pg-cdc-connector"
  },
  "tasks": [],
  "type": "source"
}


In [4]:
time.sleep(10)

r = requests.get(f"{CONNECT_URL}/connectors/pg-cdc-connector/status")
status = r.json()
print(f"Connector: {status['connector']['state']}")
for task in status.get('tasks', []):
    print(f"Task {task['id']}: {task['state']}")

assert status['connector']['state'] == 'RUNNING', f"Connector not running: {status}"

Connector: RUNNING
Task 0: RUNNING


In [5]:
admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP)
topics = admin.list_topics()
admin.close()

print("Kafka topics:")
for t in sorted(topics):
    print(f"  {t}")

assert "dbserver1.public.customers" in topics, "CDC topic not found! Check connector status."
assert "dbserver1.public.drivers" in topics, "CDC topic not found! Check connector status."
print("\n✓ CDC topic 'dbserver1.public.customers' exists!")
print("\n✓ CDC topic 'dbserver1.public.drivers' exists!")

Kafka topics:
  __consumer_offsets
  _connect_configs
  _connect_offsets
  _connect_statuses
  dbserver1.public.customers
  dbserver1.public.drivers
  taxi-trips

✓ CDC topic 'dbserver1.public.customers' exists!

✓ CDC topic 'dbserver1.public.drivers' exists!


In [6]:
# Parse Debezium CDC events from Kafka DataFrame
def parse_customer_cdc_events(kafka_df):
    """
    Parse Debezium CDC events from Kafka DataFrame.
    Extracts metadata and payload fields for customers.
    """
    # Select Kafka metadata and raw value
    raw = kafka_df.select(
        F.col("offset").alias("kafka_offset"),
        F.col("partition").alias("kafka_partition"),
        F.col("timestamp").alias("kafka_timestamp"),
        F.col("value").cast("string").alias("raw_value"),
    ).filter(F.col("raw_value").isNotNull())
    
    # Parse JSON envelope - extract payload fields
    parsed = raw.select(
        # Kafka metadata
        "kafka_offset",
        "kafka_partition", 
        "kafka_timestamp",
        # Debezium op code
        F.get_json_object("raw_value", "$.payload.op").alias("op"),
        # Before state (null for inserts)
        F.get_json_object("raw_value", "$.payload.before.id").cast("int").alias("before_id"),
        F.get_json_object("raw_value", "$.payload.before.name").alias("before_name"),
        F.get_json_object("raw_value", "$.payload.before.email").alias("before_email"),
        F.get_json_object("raw_value", "$.payload.before.country").alias("before_country"),
        # After state (null for deletes)
        F.get_json_object("raw_value", "$.payload.after.id").cast("int").alias("after_id"),
        F.get_json_object("raw_value", "$.payload.after.name").alias("after_name"),
        F.get_json_object("raw_value", "$.payload.after.email").alias("after_email"),
        F.get_json_object("raw_value", "$.payload.after.country").alias("after_country"),
        # Tombstone handling
        #F.when(F.col("raw_value").isNull(), "tombstone").otherwise("event").alias("event_type"),
        # Source metadata
        F.get_json_object("raw_value", "$.payload.source.lsn").cast("long").alias("source_lsn"),
        F.get_json_object("raw_value", "$.payload.ts_ms").cast("long").alias("ts_ms"),
    )
    
    return parsed

def parse_driver_cdc_events(kafka_df):
    """
    Parse Debezium CDC events from Kafka DataFrame for drivers table.
    Extracts metadata and payload fields for drivers.
    """
    # Select Kafka metadata and raw value
    raw = kafka_df.select(
        F.col("offset").alias("kafka_offset"),
        F.col("partition").alias("kafka_partition"),
        F.col("timestamp").alias("kafka_timestamp"),
        F.col("value").cast("string").alias("raw_value"),
    ).filter(F.col("raw_value").isNotNull())
    
    # Parse JSON envelope - extract payload fields
    parsed = raw.select(
        # Kafka metadata
        "kafka_offset",
        "kafka_partition", 
        "kafka_timestamp",
        # Debezium op code
        F.get_json_object("raw_value", "$.payload.op").alias("op"),
        # Before state (null for inserts)
        F.get_json_object("raw_value", "$.payload.before.id").cast("int").alias("before_id"),
        F.get_json_object("raw_value", "$.payload.before.name").alias("before_name"),
        F.get_json_object("raw_value", "$.payload.before.license_number").alias("before_license_number"),
        F.get_json_object("raw_value", "$.payload.before.rating").alias("before_rating"),
        F.get_json_object("raw_value", "$.payload.before.city").alias("before_city"),
        F.get_json_object("raw_value", "$.payload.before.active").cast("boolean").alias("before_active"),
        # After state (null for deletes)
        F.get_json_object("raw_value", "$.payload.after.id").cast("int").alias("after_id"),
        F.get_json_object("raw_value", "$.payload.after.name").alias("after_name"),
        F.get_json_object("raw_value", "$.payload.after.license_number").alias("after_license_number"),
        F.get_json_object("raw_value", "$.payload.after.rating").alias("after_rating"),
        F.get_json_object("raw_value", "$.payload.after.city").alias("after_city"),
        F.get_json_object("raw_value", "$.payload.after.active").cast("boolean").alias("after_active"),
        # Source metadata
        F.get_json_object("raw_value", "$.payload.source.lsn").cast("long").alias("source_lsn"),
        F.get_json_object("raw_value", "$.payload.ts_ms").cast("long").alias("ts_ms"),
    )
    
    return parsed


In [7]:
# Initial snapshot and verify that CDC events appear in Kafka topics
customers_raw = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", "dbserver1.public.customers")
    .option("startingOffsets", "earliest")
    .load()
)

customers_parsed = parse_customer_cdc_events(customers_raw)

print("Customer CDC operation counts:")
customers_parsed.groupBy("op").count().show()

print("Sample snapshot (op = 'r') events:")
customers_parsed.filter(F.col("op") == "r").select(
    "kafka_offset",
    "op",
    "after_id",
    "after_name",
    "after_email",
    "after_country"
).show(10, truncate=False)

drivers_raw = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", "dbserver1.public.drivers")
    .option("startingOffsets", "earliest")
    .load()
)

drivers_parsed = parse_driver_cdc_events(drivers_raw)

print("Driver CDC operation counts:")
drivers_parsed.groupBy("op").count().show()

print("Sample snapshot (op = 'r') events:")
drivers_parsed.filter(F.col("op") == "r").select(
    "kafka_offset",
    "op",
    "after_id",
    "after_name",
    "after_license_number",
    "after_city",
    "after_rating"
).show(10, truncate=False)

Customer CDC operation counts:
+---+-----+
| op|count|
+---+-----+
|  r|   11|
|  c|   93|
|  u|   95|
|  d|   55|
+---+-----+

Sample snapshot (op = 'r') events:
+------------+---+--------+--------------+-----------------------+-------------+
|kafka_offset|op |after_id|after_name    |after_email            |after_country|
+------------+---+--------+--------------+-----------------------+-------------+
|0           |r  |3       |Carol Ozols   |carol@example.com      |Latvia       |
|1           |r  |5       |Eva Svensson  |eva@example.com        |Sweden       |
|2           |r  |6       |Frank Muller  |frank@example.com      |Germany      |
|3           |r  |11      |Maria Silva   |maria.silva@mail.com   |India        |
|4           |r  |8       |Hiro Tanaka   |hiro@example.com       |Spain        |
|5           |r  |12      |Chen Silva    |chen.silva@inbox.org   |Netherlands  |
|6           |r  |7       |Grace Kim     |updated_7_496@mail.com |South Korea  |
|7           |r  |13      |

# 2. Bronze CDC layer

In [8]:
# Create Iceberg database if not exists
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.cdc")
spark.sql("USE lakehouse.cdc")

spark.sql("""
    CREATE OR REPLACE TABLE bronze_customers (
        kafka_offset        LONG,
        kafka_partition     INT,
        kafka_timestamp     TIMESTAMP,
        op                  STRING,
        before_id           INT,
        before_name         STRING,
        before_email        STRING,
        before_country      STRING,
        after_id            INT,
        after_name          STRING,
        after_email         STRING,
        after_country       STRING,
        source_lsn          LONG,
        ts_ms               LONG
    ) USING iceberg
""")

spark.sql("""
    CREATE OR REPLACE TABLE bronze_drivers (
        kafka_offset            LONG,
        kafka_partition         INT,
        kafka_timestamp         TIMESTAMP,
        op                      STRING,
        before_id               INT,
        before_name             STRING,
        before_license_number   STRING,
        before_rating           STRING,
        before_city             STRING,
        before_active           BOOLEAN,
        after_id                INT,
        after_name              STRING,
        after_license_number    STRING,
        after_rating            STRING,
        after_city              STRING,
        after_active            BOOLEAN,
        source_lsn              LONG,
        ts_ms                   LONG
    ) USING iceberg
""")

print("Bronze tables created on MinIO")

Bronze tables created on MinIO


In [9]:
customers_raw = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", "dbserver1.public.customers")
    .option("startingOffsets", "earliest")
    .load()
)

customers_bronze = parse_customer_cdc_events(customers_raw)
customers_bronze.writeTo("bronze_customers").append()

print(f"Bronze: {spark.table('bronze_customers').count()} CDC events loaded")
spark.sql("""
    SELECT kafka_offset, op, 
           COALESCE(after_id, before_id) AS id,
           COALESCE(after_name, before_name) AS name,
           COALESCE(after_email, before_email) AS email
    FROM bronze_customers
    ORDER BY kafka_offset
""").show(truncate=False)

Bronze: 258 CDC events loaded
+------------+---+---+--------------+------------------------+
|kafka_offset|op |id |name          |email                   |
+------------+---+---+--------------+------------------------+
|0           |r  |3  |Carol Ozols   |carol@example.com       |
|1           |r  |5  |Eva Svensson  |eva@example.com         |
|2           |r  |6  |Frank Muller  |frank@example.com       |
|3           |r  |11 |Maria Silva   |maria.silva@mail.com    |
|4           |r  |8  |Hiro Tanaka   |hiro@example.com        |
|5           |r  |12 |Chen Silva    |chen.silva@inbox.org    |
|6           |r  |7  |Grace Kim     |updated_7_496@mail.com  |
|7           |r  |13 |Anna Mets     |anna.mets@inbox.org     |
|8           |r  |14 |Sofia Virtanen|sofia.virtanen@test.net |
|9           |r  |2  |Bob Virtanen  |bob@example.com         |
|10          |r  |10 |Javier Garcia |javier@example.com      |
|11          |c  |15 |Ivan Ozols    |ivan.ozols@test.net     |
|12          |u  |15 |Iva

In [10]:
spark.sql("SELECT count(*) FROM lakehouse.cdc.bronze_customers").show()

spark.sql("""
  SELECT op, after_id, after_name, after_email, ts_ms
  FROM lakehouse.cdc.bronze_customers
  LIMIT 5
""").show(truncate=False)

# Bronze is append only to preserve the full history of CDC events as they arrive, so no data is lost and
# downstream layers can correctly reconstruct changes without modifying or deleting original records.

+--------+
|count(1)|
+--------+
|     258|
+--------+

+---+--------+------------+--------------------+-------------+
|op |after_id|after_name  |after_email         |ts_ms        |
+---+--------+------------+--------------------+-------------+
|r  |3       |Carol Ozols |carol@example.com   |1776347305859|
|r  |5       |Eva Svensson|eva@example.com     |1776347305861|
|r  |6       |Frank Muller|frank@example.com   |1776347305861|
|r  |11      |Maria Silva |maria.silva@mail.com|1776347305861|
|r  |8       |Hiro Tanaka |hiro@example.com    |1776347305861|
+---+--------+------------+--------------------+-------------+



In [11]:
drivers_raw = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", "dbserver1.public.drivers")
    .option("startingOffsets", "earliest")
    .load()
)

drivers_bronze = parse_driver_cdc_events(drivers_raw)
drivers_bronze.writeTo("bronze_drivers").append()

print(f"Bronze: {spark.table('bronze_drivers').count()} CDC events loaded")
spark.sql("""
    SELECT 
    kafka_offset,
    op,
    COALESCE(after_id, before_id) AS id,
    COALESCE(after_name, before_name) AS name,
    COALESCE(after_license_number, before_license_number) AS license_number,
    COALESCE(after_city, before_city) AS city,

    -- decode rating inline (actually keeping raw Base64 as string in bronze, decoding in silver)
    CASE 
        WHEN COALESCE(after_rating, before_rating) IS NOT NULL THEN
            CAST(conv(hex(unbase64(COALESCE(after_rating, before_rating))), 16, 10) AS INT) / 100.0
    END AS rating

FROM lakehouse.cdc.bronze_drivers
ORDER BY kafka_offset
""").show(truncate=False)

Bronze: 192 CDC events loaded
+------------+---+---+------------+--------------+-------------+--------+
|kafka_offset|op |id |name        |license_number|city         |rating  |
+------------+---+---+------------+--------------+-------------+--------+
|0           |r  |3  |Mike Road   |TLC-10003     |Queens       |4.910000|
|1           |r  |5  |Jake Taxi   |TLC-10005     |Manhattan    |4.550000|
|2           |r  |6  |Nina Cab    |TLC-10006     |Brooklyn     |4.930000|
|3           |r  |7  |Omar Ride   |TLC-10007     |Queens       |4.800000|
|4           |r  |8  |Priya Fast  |TLC-10008     |Manhattan    |4.770000|
|5           |r  |4  |Lisa Lane   |TLC-10004     |Bronx        |4.680000|
|6           |r  |1  |Tom Driver  |TLC-10001     |Manhattan    |4.850000|
|7           |r  |9  |Lars Ozols  |TLC-87014     |Staten Island|4.840000|
|8           |r  |2  |Sarah Wheels|TLC-10002     |Brooklyn     |3.730000|
|9           |u  |4  |Lisa Lane   |TLC-10004     |Bronx        |4.680000|
|10     

# 3. Silver layer

In [12]:
# Create silver tables
spark.sql("""
  CREATE TABLE IF NOT EXISTS lakehouse.cdc.silver_customers (
    id               INT,
    name             STRING,
    email            STRING,
    country          STRING,
    last_updated_ms  BIGINT
  ) USING iceberg
""")

spark.sql("""
  CREATE TABLE IF NOT EXISTS lakehouse.cdc.silver_drivers (
    id               INT,
    name             STRING,
    license_number   STRING,
    city             STRING,
    rating           DECIMAL(3,2),
    last_updated_ms  BIGINT
  ) USING iceberg
""")

print("Silver tables ready")

w = Window.partitionBy("entity_id").orderBy(F.col("ts_ms").desc())

Silver tables ready


In [13]:
# Customers
bronze_customers = spark.table("lakehouse.cdc.bronze_customers")

deduped_customers = (
    bronze_customers
    .filter(F.col("op").isNotNull())
    .withColumn("entity_id", F.coalesce(F.col("after_id"), F.col("before_id")))
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .localCheckpoint()
)

print(f"Deduplicated CDC events: {deduped_customers.count()} rows (one per primary key)")
deduped_customers.select("entity_id", "op", "after_name", "ts_ms").show(10, truncate=False)

deduped_customers.createOrReplaceTempView("cdc_batch")

spark.sql("""
  MERGE INTO lakehouse.cdc.silver_customers AS t
  USING cdc_batch AS s
  ON t.id = s.entity_id
  WHEN MATCHED AND s.op = 'd' THEN DELETE
  WHEN MATCHED AND s.op IN ('c', 'u', 'r') THEN UPDATE SET
    t.name             = s.after_name,
    t.email            = s.after_email,
    t.country          = s.after_country,
    t.last_updated_ms  = s.ts_ms
  WHEN NOT MATCHED AND s.op != 'd' THEN INSERT
    (id, name, email, country, last_updated_ms)
    VALUES (s.after_id, s.after_name, s.after_email, s.after_country, s.ts_ms)
""")

print("MERGE complete")
# If the MERGE is run again without new data, the silver table does not change 
# because the latest state per key is already applied, making the operation idempotent.
spark.sql("SELECT count(*) FROM lakehouse.cdc.silver_customers").show()

spark.sql("""
SELECT * FROM lakehouse.cdc.silver_customers
LIMIT 5
""").show(truncate=False)

pg_execute("SELECT COUNT(*) AS n FROM customers;", fetch=True)

Deduplicated CDC events: 104 rows (one per primary key)
+---------+---+-----------+-------------+
|entity_id|op |after_name |ts_ms        |
+---------+---+-----------+-------------+
|2        |d  |NULL       |1776347490626|
|3        |u  |Carol Ozols|1776347571543|
|5        |d  |NULL       |1776347342369|
|6        |d  |NULL       |1776347448649|
|7        |d  |NULL       |1776347548282|
|8        |d  |NULL       |1776347471410|
|10       |d  |NULL       |1776347315055|
|11       |d  |NULL       |1776347332255|
|12       |d  |NULL       |1776347320114|
|13       |d  |NULL       |1776347319102|
+---------+---+-----------+-------------+
only showing top 10 rows
MERGE complete
+--------+
|count(1)|
+--------+
|      49|
+--------+

+---+--------------+-------------------------+-----------+---------------+
|id |name          |email                    |country    |last_updated_ms|
+---+--------------+-------------------------+-----------+---------------+
|3  |Carol Ozols   |updated_3_309@m

[(49,)]

In [14]:
# Drivers
bronze_drivers = spark.table("lakehouse.cdc.bronze_drivers")

deduped_drivers = (
    bronze_drivers
    .filter(F.col("op").isNotNull())
    .withColumn("entity_id", F.coalesce(F.col("after_id"), F.col("before_id")))
    .withColumn("raw_rating", F.coalesce(F.col("after_rating"), F.col("before_rating")))
    .withColumn(
        "rating",
        F.conv(
            F.hex(F.unbase64(F.coalesce(F.col("after_rating"), F.col("before_rating")))),
            16,
            10
        ).cast("int") / 100.0
    )
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .localCheckpoint()
)

print(f"Deduplicated CDC events: {deduped_drivers.count()} rows (one per primary key)")
deduped_drivers.select("entity_id", "op", "after_name", "ts_ms").show(10, truncate=False)

deduped_drivers.createOrReplaceTempView("cdc_drivers_batch")

spark.sql("""
  MERGE INTO lakehouse.cdc.silver_drivers AS t
  USING cdc_drivers_batch AS s
  ON t.id = s.entity_id
  WHEN MATCHED AND s.op = 'd' THEN DELETE
  WHEN MATCHED AND s.op IN ('c', 'u', 'r') THEN UPDATE SET
    t.name             = s.after_name,
    t.license_number   = s.after_license_number,
    t.city             = s.after_city,
    t.rating           = s.rating,
    t.last_updated_ms  = s.ts_ms
  WHEN NOT MATCHED AND s.op != 'd' THEN INSERT
    (id, name, license_number, city, rating, last_updated_ms)
    VALUES (s.after_id, s.after_name, s.after_license_number, s.after_city, s.rating, s.ts_ms)
""")

print("MERGE complete")

spark.sql("SELECT count(*) FROM lakehouse.cdc.silver_drivers").show()

spark.sql("""
SELECT * FROM lakehouse.cdc.silver_drivers
LIMIT 5
""").show(truncate=False)

pg_execute("SELECT COUNT(*) AS n FROM drivers;", fetch=True)

Deduplicated CDC events: 61 rows (one per primary key)
+---------+---+----------+-------------+
|entity_id|op |after_name|ts_ms        |
+---------+---+----------+-------------+
|1        |d  |NULL      |1776347334278|
|2        |d  |NULL      |1776347352485|
|3        |d  |NULL      |1776347430424|
|4        |d  |NULL      |1776347408676|
|5        |d  |NULL      |1776347466353|
|6        |d  |NULL      |1776347306956|
|7        |d  |NULL      |1776347436497|
|8        |d  |NULL      |1776347307968|
|9        |d  |NULL      |1776347500748|
|10       |d  |NULL      |1776347322137|
+---------+---+----------+-------------+
only showing top 10 rows
MERGE complete
+--------+
|count(1)|
+--------+
|      22|
+--------+

+---+--------------+--------------+-------------+------+---------------+
|id |name          |license_number|city         |rating|last_updated_ms|
+---+--------------+--------------+-------------+------+---------------+
|19 |Mia Tanaka    |TLC-65885     |Brooklyn     |3.52  |

[(22,)]